In [2]:
!apt-get install -y gdal-bin libgdal-dev 2>/dev/null | tail -3
!pip install -q rasterio==1.3.10 shapely psycopg2-binary boto3 numpy tqdm
!pip install boto3 botocore

#### Data Transformation
!pip install -q pyspark apache-sedona
from google.colab import userdata
AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
print('AWS credentials loaded.')
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, year, avg

spark = SparkSession.builder \
    .appName("VegetationPipeline") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID) \
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .getOrCreate()
print(f"Spark initialized: {spark.version}")
# Tile N39W108 covers lat 39-42°N, lon -108 to -105°W (includes Boulder County)
!curl -L \ https://esa-worldcover.s3.eu-central-1.amazonaws.com/v100/2020/map/ESA_WorldCover_10m_2020_v100_N39W108_Map.tif \
-o /tmp/worldcover_boulder.tif
print('WorldCover tile downloaded to /tmp/worldcover_boulder.tif')
import numpy as np
import boto3
import rasterio
from rasterio.session import AWSSession
from rasterio.transform import Affine

def read_bands_from_s3(results, aws_key, aws_secret, region='us-west-1',
                        red_band=3, nir_band=4, swir1_band=5, pixel_stride=10):
    """
    Reads Red, NIR, and SWIR1 band arrays directly from S3 COGs for each
    scene in results. Streams via rasterio AWSSession — no local download.

    Returns list of (tile_id, acq_date, red, nir, swir1, bbox_wkt,
                      strided_transform, crs_str) per scene.
    strided_transform and crs_str are needed by apply_forest_mask.
    """
    boto3_session = boto3.Session(
        aws_access_key_id=aws_key,
        aws_secret_access_key=aws_secret,
        region_name=region,
    )
    aws_session = AWSSession(boto3_session)
    scene_bands = []
    for tile_id, acq_date, file_url, _, _res, bbox_wkt in results:
        print(f'  Reading {tile_id}  ->  {file_url}')
        with rasterio.Env(aws_session):
            with rasterio.open(file_url) as src:
                n = src.count
                print(f'    bands={n}  shape={src.height}x{src.width}  crs={src.crs}')
                rb = min(red_band,   n)
                nb = min(nir_band,   n)
                sb = min(swir1_band, n)
                red   = src.read(rb)[::pixel_stride, ::pixel_stride].astype('float32')
                nir   = src.read(nb)[::pixel_stride, ::pixel_stride].astype('float32')
                swir1 = src.read(sb)[::pixel_stride, ::pixel_stride].astype('float32')
                nodata = src.nodata if src.nodata is not None else 0
                for arr in (red, nir, swir1):
                    arr[arr == nodata] = np.nan
                # Adjust affine transform for the stride so apply_forest_mask
                # can reproject WorldCover onto this exact subsampled grid
                t = src.transform
                strided_transform = Affine(
                    t.a * pixel_stride, t.b, t.c,
                    t.d, t.e * pixel_stride, t.f,
                )
                crs_str = src.crs
        print(f'    red   [{np.nanmin(red):.0f}, {np.nanmax(red):.0f}]')
        print(f'    nir   [{np.nanmin(nir):.0f}, {np.nanmax(nir):.0f}]')
        print(f'    swir1 [{np.nanmin(swir1):.0f}, {np.nanmax(swir1):.0f}]')
        scene_bands.append((tile_id, acq_date, red, nir, swir1, bbox_wkt,
                             strided_transform, crs_str))
    return scene_bands

import datetime
import psycopg2
from psycopg2 import sql

# Redefine necessary variables for CONFIG, including userdata calls
from google.colab import userdata
import os

AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
AWS_REGION            = 'us-west-1'
S3_BUCKET_NAME        = userdata.get('S3_BUCKET_NAME')

DB_CONFIG = {
    'host':     userdata.get('DB_HOST'),
    'port':     5432,
    'database': 'postgres',
    'user':     'postgres',
    'password': userdata.get('DB_PASSWORD'),
    'connect_timeout': 10
}
S3_RAW_PREFIX = 'raw/'

LOCAL_WORK_DIR = '/tmp/vegetation_pipeline'
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

# Placeholder for DRIVE_SOURCE_DIR, as it's not strictly needed for this cell's function
# and is commented out in its original cell.
DRIVE_SOURCE_DIR = '/content/drive/MyDrive/BigDataProject' # Assuming this default path if needed elsewhere

# Redefine CONFIG dictionary
CONFIG = {
    'aws_access_key_id':     AWS_ACCESS_KEY_ID,
    'aws_secret_access_key': AWS_SECRET_ACCESS_KEY,
    'aws_region':            AWS_REGION,
    's3_bucket':             S3_BUCKET_NAME,
    's3_cog_prefix':         'cog/',
    's3_raw_prefix':         S3_RAW_PREFIX,
    'db_config':             DB_CONFIG,
    'phase':                 1, # Defaulting to phase 1, though not directly used here.
    'drive_source_dir':      DRIVE_SOURCE_DIR,
    'local_work_dir':        LOCAL_WORK_DIR,
    'upload_to_s3':          True,
    'skip_existing':         True,
    'cog_compress':          'LZW',
    'cog_blocksize':         512,
    'cog_overviews':         [2, 4, 8, 16],
    'compute_ndvi_preview':  True,
    'ndvi_sample_size':      1000,
}

def get_db_connection(config):
    db_config = config['db_config']
    try:
        conn = psycopg2.connect(
            host=db_config['host'],
            port=db_config['port'],
            database=db_config['database'],
            user=db_config['user'],
            password=db_config['password'],
            connect_timeout=db_config.get('connect_timeout', 10)
        )
        conn.autocommit = False   # Use explicit transactions

        # Test PostGIS availability
        cur = conn.cursor()
        cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")
        cur.execute("SELECT PostGIS_Version();")
        postgis_ver = cur.fetchone()[0]
        cur.execute("SELECT version();")
        pg_ver = cur.fetchone()[0].split(' ')[1]

        print(f"  Database connected successfully!")
        print(f"  PostgreSQL version: {pg_ver}")
        print(f"  PostGIS version: {postgis_ver.split(' ')[0]}")
        cur.close()
        return conn

    except psycopg2.OperationalError as e:
        print(f"  [ERROR] Cannot connect to database: {e}")
        print(f"  Check: 1) RDS endpoint correct? 2) Security group allows port 5432?")
        print(f"         3) RDS instance is 'Available' in AWS console?")
        return None

def run_spatiotemporal_query(conn, west, south, east, north,
                              start_date, end_date):
    cur = conn.cursor()
    cur.execute("""
        SELECT
            tile_id,
            acquisition_date,
            file_url,
            ndvi_mean,
            resolution_m,
            ST_AsText(geom) AS bbox_wkt
        FROM vegetation_metadata
        WHERE
            ST_Intersects(
                geom,
                ST_MakeEnvelope(%s, %s, %s, %s, 4326)
            )
            AND acquisition_date BETWEEN %s AND %s
        ORDER BY acquisition_date;
    """, (west, south, east, north, start_date, end_date))

    results = cur.fetchall()
    cur.close()
    return results

# Establish DB connection using the CONFIG dictionary defined earlier
conn_transform = get_db_connection(config=CONFIG)
if conn_transform:
    # Query for all COG URLs. Use a very wide spatial and temporal range
    # to capture all indexed scenes.
    all_bounds = (-180.0, -90.0, 180.0, 90.0) # Global extent
    all_start_date = datetime.date(1900, 1, 1) # Start of modern remote sensing
    all_end_date   = datetime.date.today() + datetime.timedelta(days=365) # Extend to future proof

    print("Fetching all COG URLs from PostGIS database...")
    results = run_spatiotemporal_query(
        conn_transform,
        *all_bounds,
        all_start_date, all_end_date
    )
    conn_transform.close()
    print(f'Found {len(results)} scene(s) in the database.')
else:
    print('Failed to connect to database. Cannot fetch scene URLs.')
    results = [] # Ensure results is an empty list if DB connection fails

if results:
    scene_bands = read_bands_from_s3(
        results,
        AWS_ACCESS_KEY_ID,
        AWS_SECRET_ACCESS_KEY,
        region=AWS_REGION,
        red_band=1, nir_band=2, swir1_band=3,  # Adjusted band assignments
        pixel_stride=10,   # set to 1 for full resolution
    )
    print(f'Bands loaded from S3 for {len(scene_bands)} scene(s). No download needed.')
else:
    print('No results found or database connection failed. Skipping band reading.')
    scene_bands = [] # Ensure scene_bands is defined.
def build_pixel_dataframe(sedona, scene_bands):
    """
    Flattens per-scene band arrays into a Spark DataFrame with one row per pixel.
    Columns: tile_id, date, red, nir, swir1, pixel_id, bbox_wkt
    NaN pixels (incl. non-forest pixels zeroed by apply_forest_mask) are dropped.
    Accepts both 6- and 8-tuple scene entries — extra fields (transform, crs) ignored.
    """
    pixel_rows = []
    for scene in scene_bands:
        tile_id, acq_date, red, nir, swir1, bbox_wkt = scene[:6]
        h, w = red.shape
        for r in range(h):
            for c in range(w):
                rv, nv, sv = float(red[r, c]), float(nir[r, c]), float(swir1[r, c])
                if rv != rv or nv != nv or sv != sv:  # NaN check
                    continue
                pixel_rows.append((
                    tile_id, acq_date,
                    rv, nv, sv,
                    f'{tile_id}_{r}_{c}',
                    bbox_wkt,
                ))
    print(f'  Pixel rows (non-NaN): {len(pixel_rows):,}')
    return sedona.createDataFrame(
        pixel_rows,
        ['tile_id', 'date', 'red', 'nir', 'swir1', 'pixel_id', 'bbox_wkt'],
    )

from rasterio.warp import reproject, Resampling

def apply_forest_mask(scene_bands, worldcover_path, forest_class=10):
    """
    Masks each scene's band arrays to forested pixels only using ESA WorldCover.
    Class 10 = Tree cover. Non-forest pixels are set to NaN.

    Operates on numpy arrays (before Spark) by reprojecting the WorldCover
    GeoTIFF onto each scene's subsampled pixel grid using rasterio.
    No Sedona, no geoparquet, no geom column required.

    Parameters
    ----------
    scene_bands     : list of 8-tuples from read_bands_from_s3()
                      (..., strided_transform, crs_str)
    worldcover_path : local path to the ESA WorldCover GeoTIFF
    forest_class    : WorldCover class for tree cover (default 10)
    """
    masked = []
    with rasterio.open(worldcover_path) as wc:
        for scene in scene_bands:
            tile_id, acq_date, red, nir, swir1, bbox_wkt, transform, crs = scene
            h, w = red.shape
            forest_mask = np.zeros((h, w), dtype=np.uint8)
            reproject(
                source=rasterio.band(wc, 1),
                destination=forest_mask,
                dst_transform=transform,
                dst_crs=crs,
                resampling=Resampling.nearest,
            )
            is_forest = (forest_mask == forest_class)
            for arr in (red, nir, swir1):
                arr[~is_forest] = np.nan
            n_forest = int(is_forest.sum())
            print(f'  [{tile_id}] Forest pixels: {n_forest:,} / {h*w:,} '
                  f'({100*n_forest/max(h*w,1):.1f}%)')
            masked.append((tile_id, acq_date, red, nir, swir1, bbox_wkt, transform, crs))
    return masked

def compute_vegetation_indices(df):
    """
    Computes NDVI and NDMI from Red, NIR, and SWIR1 bands.
    NDVI = (NIR - Red)  / (NIR + Red)
    NDMI = (NIR - SWIR1) / (NIR + SWIR1)
    """
    return df.withColumn('ndvi', (col('nir') - col('red'))   / (col('nir') + col('red'))) \
             .withColumn('ndmi', (col('nir') - col('swir1')) / (col('nir') + col('swir1')))

def harmonize_time_series(df):
    """
    Aggregates irregular observations into monthly composites.
    Returns a regular time series for each pixel.
    """
    return (
        df.groupBy('pixel_id', year('date').alias('year'), month('date').alias('month'))
          .agg(avg('ndvi').alias('ndvi_mean'), avg('ndmi').alias('ndmi_mean'))
          .orderBy('year', 'month')
    )

if 'scene_bands' in locals() and scene_bands:
    forest_data_path = '/tmp/worldcover_boulder.tif'
    masked_bands = apply_forest_mask(scene_bands, forest_data_path)
    spark_df   = build_pixel_dataframe(spark, masked_bands)
    indices_df = compute_vegetation_indices(spark_df)
    final_df   = harmonize_time_series(indices_df)
    final_df.show(10)
    print(f'Transformation Complete: {final_df.count()} monthly records generated.')
else:
    print('No scene_bands found. Run the S3 band-read cell above first.')
final_df.head(5)

# Z-Score Anomaly Detection
from pyspark.sql.functions import stddev, mean as spark_mean, when, col

# Historical baseline (mean & std per pixel per month)
baseline_df = final_df.groupBy('pixel_id', 'month').agg(
    spark_mean('ndvi_mean').alias('ndvi_mu'),
    stddev('ndvi_mean').alias('ndvi_sigma'),
    spark_mean('ndmi_mean').alias('ndmi_mu'),
    stddev('ndmi_mean').alias('ndmi_sigma'),
)

# Join with baseline and compute Z-scores
anomaly_df = final_df.join(baseline_df, on=['pixel_id', 'month'])

anomaly_df = anomaly_df.withColumn(
    'ndvi_zscore',
    when(col('ndvi_sigma') > 0,
         (col('ndvi_mean') - col('ndvi_mu')) / col('ndvi_sigma'))
    .otherwise(0.0)
).withColumn(
    'ndmi_zscore',
    when(col('ndmi_sigma') > 0,
         (col('ndmi_mean') - col('ndmi_mu')) / col('ndmi_sigma'))
    .otherwise(0.0)
)

# Flag anomalies (Z < -2.0)
THRESHOLD = -1.5
anomaly_df = anomaly_df.withColumn(
    'is_ndvi_anomaly', col('ndvi_zscore') < THRESHOLD
).withColumn(
    'is_ndmi_anomaly', col('ndmi_zscore') < THRESHOLD
)

# Show flagged anomalies
anomalies = anomaly_df.filter(col('is_ndvi_anomaly') | col('is_ndmi_anomaly'))
anomalies.show(20)
print(f'Total anomalies detected: {anomalies.count()}')
print(f'Total records evaluated: {anomaly_df.count()}')

Setting up python3-gdal (3.8.4+dfsg-1~jammy0) ...
Setting up gdal-bin (3.8.4+dfsg-1~jammy0) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 42.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 58.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 74.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.13.0 requires rasterio>=1.4, but you have rasterio 1.3.10 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.9/331.9 kB 16.6 MB/s eta 0:00:00


TimeoutException: Requesting secret AWS_ACCESS_KEY_ID timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# Export anomaly_events parquet to S3
from datetime import datetime
from pyspark.sql.functions import lit, regexp_extract, col

if 'anomaly_df' not in locals():
    raise ValueError('anomaly_df is not defined. Run the anomaly detection cell first.')
if 'results' not in locals() or not results:
    raise ValueError('results is not available. Run the PostGIS fetch step first.')
if 'S3_BUCKET_NAME' not in locals() or not S3_BUCKET_NAME:
    raise ValueError('S3_BUCKET_NAME is missing.')

# Toggle: "full" or "sample"
EXPORT_MODE = 'full'
SAMPLE_FRACTION = 0.10
SAMPLE_SEED = 42

run_id = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
output_path = f"s3://{S3_BUCKET_NAME}/anomaly_events/run_id={run_id}"

# Build tile_id -> source_s3_uri mapping from query results
result_uri_rows = [(r[0], r[2]) for r in results]
result_uri_df = spark.createDataFrame(result_uri_rows, ['tile_id', 'source_s3_uri']).dropDuplicates(['tile_id'])

# Recover tile_id from pixel_id pattern: <tile_id>_<row>_<col>
anomaly_events_base = anomaly_df.withColumn(
    'tile_id',
    regexp_extract(col('pixel_id'), r'^(.*)_\\d+_\\d+$', 1),
)

anomaly_events = anomaly_events_base.join(result_uri_df, on='tile_id', how='left').select(
    lit(run_id).alias('run_id'),
    col('pixel_id'),
    col('year').cast('int').alias('year'),
    col('month').cast('int').alias('month'),
    col('ndvi_zscore').cast('double').alias('ndvi_zscore'),
    col('ndmi_zscore').cast('double').alias('ndmi_zscore'),
    col('is_ndvi_anomaly').cast('boolean').alias('is_ndvi_anomaly'),
    col('is_ndmi_anomaly').cast('boolean').alias('is_ndmi_anomaly'),
    col('source_s3_uri'),
)

if EXPORT_MODE == 'sample':
    anomaly_events_to_write = anomaly_events.sample(False, SAMPLE_FRACTION, SAMPLE_SEED)
else:
    anomaly_events_to_write = anomaly_events

# Write parquet
anomaly_events_to_write.repartition(8).write.mode('overwrite').parquet(output_path)

# Verify write
written_df = spark.read.parquet(output_path)
print(f'Wrote anomaly_events parquet to: {output_path}')
print(f'Rows written: {written_df.count()}')
written_df.printSchema()
written_df.show(10, truncate=False)